In [ ]:
# #Used to test that document parser py is working properly. 

# from document_parser import DocumentParser

# out_dir = r"datasets\processed"
# dp = DocumentParser(out_dir=out_dir)

# file_path = r"datasets/raw/COF_HY24_IP.pdf"

# dp.ingest_document(file_path=file_path)

c:\Users\chuak\OneDrive\Documents\Projects\LLM-based-information-extraction-for-financial-documents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing file: COF_HY24_IP.pdf


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[INFO] 2026-02-08 16:27:43,214 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-08 16:27:43,219 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-02-08 16:27:43,229 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\chuak\OneDrive\Documents\Projects\LLM-based-information-extraction-for-financial-documents\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-08 16:27:43,230 [RapidOCR] main.py:50: Using C:\Users\chuak\OneDrive\Documents\Projects\LLM-based-information-extraction-for-financial-documents\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-08 16:27:43,646 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-08 16:27:43,648 [Rapi

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import openai
import numpy as np
from pathlib import Path
from openai import OpenAI
import faiss
import json

c:\Users\chuak\OneDrive\Documents\Projects\LLM-based-information-extraction-for-financial-documents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Open json file that has been saved. 
with open(r"datasets\processed\json\COF_HY24_IP.json","r") as f:
    doc_json = json.load(f)

In [4]:
from docling_core.types.doc import DoclingDocument

doc = DoclingDocument.model_validate(doc_json)



In [5]:
doc

DoclingDocument(schema_name='DoclingDocument', version='1.9.0', name='COF_HY24_IP', origin=DocumentOrigin(mimetype='application/pdf', binary_hash=2594346747007363658, filename='COF_HY24_IP.pdf', uri=None), furniture=GroupItem(self_ref='#/furniture', parent=None, children=[], content_layer=<ContentLayer.FURNITURE: 'furniture'>, meta=None, name='_root_', label=<GroupLabel.UNSPECIFIED: 'unspecified'>), body=GroupItem(self_ref='#/body', parent=None, children=[RefItem(cref='#/pictures/0'), RefItem(cref='#/pictures/1'), RefItem(cref='#/texts/4'), RefItem(cref='#/pictures/2'), RefItem(cref='#/pictures/3'), RefItem(cref='#/texts/6'), RefItem(cref='#/pictures/4'), RefItem(cref='#/pictures/5'), RefItem(cref='#/texts/10'), RefItem(cref='#/texts/11'), RefItem(cref='#/texts/12'), RefItem(cref='#/groups/0'), RefItem(cref='#/texts/18'), RefItem(cref='#/texts/19'), RefItem(cref='#/texts/20'), RefItem(cref='#/texts/21'), RefItem(cref='#/texts/22'), RefItem(cref='#/texts/23'), RefItem(cref='#/texts/24')

In [10]:
for node in doc.iterate_items():
    print(node)
    

(PictureItem(self_ref='#/pictures/0', parent=RefItem(cref='#/body'), children=[RefItem(cref='#/texts/0'), RefItem(cref='#/texts/1'), RefItem(cref='#/texts/2')], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.PICTURE: 'picture'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=0.0, t=540.0, r=959.81884765625, b=60.4022216796875, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 0))], source=[], comments=[], captions=[], references=[], footnotes=[], image=None, annotations=[]), 1)
(PictureItem(self_ref='#/pictures/1', parent=RefItem(cref='#/body'), children=[RefItem(cref='#/texts/3')], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.PICTURE: 'picture'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=795.265625, t=48.27423095703125, r=931.4057006835938, b=21.90777587890625, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 0))], source=[], comments=[], captions=[], references=[], footnotes=[], im

In [3]:
#Open json file that has been saved. 
with open(r"datasets\processed\json\COF_HY24_IP.json","r") as f:
    doc_json = json.load(f)

data = []

#Each presentation as represented as json would have multiple text fields. 
longest_text = 0

for idx, d in enumerate(doc_json["texts"]):
    page_no = d.get('prov')[0].get('page_no')
    text = d.get('text')
    longest_text = max(longest_text,len(text.split()))
    data.append({"id":idx,"page_no":page_no,"text":text})


print("length of longest chunk by number of words is ",longest_text)

length of longest chunk by number of words is  190


In [ ]:
from openai import AsyncOpenAI
import asyncio

#Remove this when running in .py file. 
import nest_asyncio
nest_asyncio.apply()

client = AsyncOpenAI()

texts = [doc.get('text') for doc in data]

async def embed_batch(texts, model="text-embedding-3-small"):
    resp = await client.embeddings.create(
        model=model,
        input=texts
    )
    return [d.embedding for d in resp.data]


async def embed_texts_async(
    texts,
    batch_size=50,
    max_concurrency=5
):
    sem = asyncio.Semaphore(max_concurrency)

    async def process_batch(batch):
        async with sem:
            return await embed_batch(batch)

    tasks = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        tasks.append(process_batch(batch))

    results = await asyncio.gather(*tasks)

    # flatten
    embeddings = []
    for r in results:
        embeddings.extend(r)

    return np.array(embeddings, dtype="float32")


In [ ]:
#Asynchronously run asyncio. 
embeddings = asyncio.run(
    embed_texts_async(texts)
)



In [17]:
# Dimensions reflect number of chunks, embedding dimensions. 
d = embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(embeddings)

In [35]:
# Convert query into vector
query_text = "What is the capitalisation rate of COF?"

#Embed texts takes in a list and returns a list of embedded text. We provided a list but only want a single query. 
query_vec = embed_texts([query_text])[0]
#I has (n_queries, k) diemnsions
D, I = index.search(query_vec.reshape(1, -1), k=8)
relevant_docs = [data[i] for i in I[0]]

In [36]:
type(relevant_docs[0]['page_no'])

int

In [37]:
from pydantic import BaseModel
from pydantic import Field
from typing import Optional

class Metric(BaseModel):
    metric_name_doc: Optional[str] = Field(description="Exact name of metric being queried in document")
    metric_val:Optional[str] = Field(description="Value of metric being extracted")
    metric_period:Optional[str] = Field(description="Financial Period of Metric period extracted. Eg: FYXX, HYXX")
    metric_page:Optional[str] = Field(description="Page number of extraction")



In [38]:
relevant_docs

[{'id': 333, 'page_no': 18, 'text': 'COF portfolio valuation summary 2,3'},
 {'id': 81,
  'page_no': 7,
  'text': 'Based on COF closing unit price of $1.25 on 14 February 2024'},
 {'id': 50, 'page_no': 5, 'text': 'Centuria Office REIT (COF)'},
 {'id': 66, 'page_no': 7, 'text': 'COF at a glance'},
 {'id': 536, 'page_no': 31, 'text': 'Weighted average capitalisation rate'},
 {'id': 351, 'page_no': 18, 'text': '4. Weighted average capitalisation rate'},
 {'id': 352,
  'page_no': 18,
  'text': '5. Based on COF closing unit price of $1.25 on 14 February 2024'},
 {'id': 548,
  'page_no': 33,
  'text': "This presentation is provided for general information purposes only. It is not a product disclosure statement, pathfinder document or any other disclosure document for the purposes of the Corporations Act and has not been, and is not required to be, lodged with the Australian Securities and Investments Commission. It should not be relied upon by the recipient in considering the merits of COF o

In [39]:
client = OpenAI()

sample_messages = [
    {"role":"system","content":"You are a financial extractor"},
    {f"role":"user","content":f"Context:\n{relevant_docs}"},
    {f"role":"user","content":"What is the FY24 cap rate of Centuria Office REIT?"}
]

model_name = "gpt-4o-mini"

#parse is for structured output
completion = client.chat.completions.parse(
    model = model_name,
    messages = sample_messages,
    response_format=Metric
)

In [40]:
completion.choices[0].message.content

'{"metric_name_doc":"Weighted average capitalisation rate","metric_val":"Not provided","metric_period":"FY24","metric_page":"18"}'

In [ ]:
# #Create chunks with recursive text splitter
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
# chunks = text_splitter.split_text(text)